In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# 1. Load train and test files
# -----------------------------
train = pd.read_csv("model3_train.csv")
test = pd.read_csv("model3_test.csv")

# -----------------------------
# 2. Define target and features
# -----------------------------
target = "TOTAL_TRAFFIC"

X_train = train.drop(columns=[target])
y_train = train[target]

X_test = test.drop(columns=[target])
y_test = test[target]

# -----------------------------
# 3. Base models
#    Approximate H2O ensemble:
#    2 GBMs + 5 Deep Learning models
# -----------------------------
estimators = [
    ("gbm_1", GradientBoostingRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    )),
    
    ("gbm_2", GradientBoostingRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        random_state=43
    )),

    ("mlp_1", MLPRegressor(
        hidden_layer_sizes=(32,),
        activation="relu",
        solver="adam",
        max_iter=5000,
        random_state=42
    )),

    ("mlp_2", MLPRegressor(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        max_iter=5000,
        random_state=43
    )),

    ("mlp_3", MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        max_iter=5000,
        random_state=44
    )),

    ("mlp_4", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        max_iter=5000,
        random_state=45
    )),

    ("mlp_5", MLPRegressor(
        hidden_layer_sizes=(16,),
        activation="relu",
        solver="adam",
        max_iter=5000,
        random_state=46
    ))
]

# -----------------------------
# 4. Metalearner
#    H2O used GLM
#    Closest sklearn equivalent: Ridge
# -----------------------------
final_estimator = Ridge(alpha=1.0)

# -----------------------------
# 5. Stacking Regressor
# -----------------------------
stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=final_estimator,
    passthrough=False,
    cv=5,
    n_jobs=None
)

# -----------------------------
# 6. Fit model
# -----------------------------
stack_model.fit(X_train, y_train)

# -----------------------------
# 7. Predict on test set
# -----------------------------
preds = stack_model.predict(X_test)

# -----------------------------
# 8. Evaluate
# -----------------------------
mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("Model 3 Python Replication Results")
print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

# -----------------------------
# 9. Optional: compare actual vs predicted
# -----------------------------
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": preds
})

print("\nActual vs Predicted:")
print(results)

Model 3 Python Replication Results
RMSE: 419194.6906373242
MAE: 278042.9755829501
R²: 0.5631004991842229

Actual vs Predicted:
      Actual     Predicted
0   10590277  1.084623e+07
1   10683884  1.083261e+07
2   10692555  1.073667e+07
3   10195557  1.037566e+07
4   10574351  1.069072e+07
5   10084757  1.042116e+07
6   10430200  1.081116e+07
7    9220491  9.672854e+06
8    8545155  9.767589e+06
9   10049725  9.988793e+06
10   9972709  1.007782e+07
11  10704604  1.067157e+07
